# A study of 'wedges, oil and vinegar'

## Experiment 1: vanishing ratio of randomly sampled vectors 

In [1]:
# what is the ratio of randomly sampled vectors that vanish on the public key?
load("generate_UOV_instance.sage")
load("wedge_attack.sage")


def test_one_instance(n, m, o, q, can_print = False):
    pk, sk, P, oil, vinegar, O, V, gens_vector = generate_UOV_variables(n, m, o, q)
    F = GF(q)
    v = n - o
    test_it = q^n # sample all the vectors TODO differentiate between number of samples drawn for O, V and q^n

    # initialize the counters
    counter_O, counter_V, counter_random = 0, 0, 0
    counter_V2 = 0
    counter_random_2 = 0

    # convert the public key to polynomials
    pk_poly = []
    pk_poly.extend(P(gens_vector*coef_matrix*gens_vector) for coef_matrix in pk)
        
    for i in range(test_it):

        # sample vectors from the oil space, V and q^n at random
        random_O_vector = vector([F.random_element() for i in range(o)])*O
        random_V_vector = vector([F.random_element() for i in range(v)])*V
        random_vector = vector([F.random_element() for i in range(n)])

        # check if the sampled vectors vanish on the public key
        if all((poly(list(random_O_vector)) == 0) for poly in pk_poly):
            counter_O += 1
        if all((poly(list(random_V_vector)) == 0) for poly in pk_poly):
            counter_V += 1
        if all((poly(list(random_vector)) == 0) for poly in pk_poly):
            counter_random += 1

    # round the ratios to two decimals
    res_O = round(counter_O/test_it, 2)
    res_V = round(counter_V/test_it, 2)
    res_both = round(counter_random/test_it, 2)

    if can_print:
        print('random O vector \t', random_O_vector, 'evaluation \t',  "not yet implemented")
        print('random V vector \t', random_V_vector, 'evaluation \t',  "not yet implemented")
        print('random vector \t', random_vector, 'evaluation \t',  "not yet implemented")
        # p1(list(random_V_vector)) == p2(list(random_V_vector)) == 0
    return res_O, res_V, res_both


def test_multiple_instances(it, can_print=True):
    res_vec = [0, 0, 0]
    
    for i in range(it):
        temp = test_one_instance(n, m, o, q)
        res_vec[0] += temp[0]
        res_vec[1] += temp[1]
        res_vec[2] += temp[2]
        
    if can_print:
        print('ratio of sampled oil vectors that vanish on P \t \t', round(res_vec[0]/it, 4))
        print('ratio of sampled vinegar vectors that vanish on P  \t', round(res_vec[1]/it, 4), "\t expected: \t", "?") # earlier: 
        print('ratio of sampled random vectors that vanish on P \t', round(res_vec[2]/it, 4), "\t expected: \t", round(q**(-m), 4))

In [2]:
n, m, o, q = 5, 2, 2, 8
it = 1

test_multiple_instances(it)

ratio of sampled oil vectors that vanish on P 	 	 1.0
ratio of sampled vinegar vectors that vanish on P  	 0.02 	 expected: 	 ?
ratio of sampled random vectors that vanish on P 	 0.01 	 expected: 	 0.0156


## Wedge attack

In [3]:
# WEDGE ATTACK
load("wedge_attack.sage")
load("generate_macaulay.sage")

def test_wedge_attack(q, v, o, m, prune=False, educational_implementation=True, can_print=False):
    pk, sk, P, oil, vinegar, O, V, gens_vector = generate_UOV_variables(n, m, o, q)
    res, v, M, dim, timings = wedge_attack(pk, n, q, o, prune, educational_implementation, can_print)

q = 2 
v, o, m = 4, 3, 6
n = o + v

test_wedge_attack(q, v, o, m)

prune? 	 False
Time to calculate the lookup dict: 	 	 2.6941299438476562e-05
Time to calculate the idx and row dicts 	 1.2636184692382812e-05
Time to compute the col_index: 	 	 0.0003478527069091797
Is the result of the educational implementation the same as that of the built-in? Yes
time to generate Macaulay matrix: 	 0.1584930419921875
M has dimensions: 	 42 35
time compute the kernel of M 	 	 0.013173341751098633
time to build the oil space 	 	 0.00047850608825683594


## Experiment 2: testing the wedge attack

In [4]:
# EXPERIMENT 2 testing this implementation of the wedge attack
load("wedge_attack.sage")
import time

def wedge_once(n, m, o, q, prune, educational_implementation, can_print=False):
    timings = []
    time_generate_UOV_variables_start = time.time()
    pk, sk, P, oil, vinegar, O, V, gens_vector = generate_UOV_variables(n, m, o, q)
    time_generate_UOV_variables_delta = time.time() - time_generate_UOV_variables_start
    timings.append(("building a UOV instance", time_generate_UOV_variables_delta))

    x_vec = vector(P.gens())
    retrieved_space, kernel_vector, M, dim, timings_wedge_attack = wedge_attack(pk, n, q, o, prune, educational_implementation, can_print=False)
  
    timings.extend(timings_wedge_attack)
    
    if can_print:
        if dim != 1:
            print(f"De kernel heeft rang {dim}")
        # print("vinegar_coefficient: \t", vinegar_coefficient)
        print(f"O in reduced echelon form: \n{O.rref()}")
        assert O.rank() == o
        print("the Macaulay matrix has rank: \t", M.rank(), f" and dimensions:\t{M.nrows()}, {M.ncols()}")
        # print("the kernel vector: \t", kernel_vector[1]) # prints only the second element. We expect the first element to be the zero matrix
        print("-----------"*7)
        print(f"the predicted rank of the kernel is: \t {predict_rank(n, m, o)}, its observed rank is: \t {dim}")
        print("-----------"*7)
        print(f"the retrieved oil space in rref: \t \n{retrieved_space.rref()}")
        print("-----------"*7)
        if O.rref() == retrieved_space.rref():
            print("O.rref == retrieved_space.rref(), attack succeeded.")
        print("-----------"*7)
        # for timing in timings:
        #     print(timing[0], "\t", timing[1])
            
    return dim, O.rref(), retrieved_space.rref(), timings

def wedge_party(q, v, o, m, test_it, prune, educational_implementation, can_print=False):
    n = v + o
    if is_admissible(n, m, o):
        counter_rank = 0
        counter_too_high = 0
        counter_attack = 0
        for i in range(test_it):
            exp, O, retrieved, timings = wedge_once(n, m, o, q, prune, educational_implementation, can_print)
            if exp == 1:
                counter_rank += 1
            if exp > 1:
                counter_too_high += 1
            if O == retrieved:
                counter_attack += 1
            # print("rank prediction", predict_rank(n, m, o))
            print(f"iteratie {i}")
        print("aandeel van de testen waar de rang gelijk is aan 1: \t", round(counter_rank/test_it, 2))
        print("aandeel van de testen waar de rang groter is dan 1: \t", round(counter_too_high/test_it, 2))
        print("aandeel van de testen waar O.rref() == res.rref(): \t", round(counter_attack/test_it, 2))
     
    else: 
        print("parameters not admissible")
        if not min((o-1)/2, 2)*m > v:
            print("not min((o-1)/2, 2)*m > v")
        if not v > o:
            print("not v > o")
        if predict_rank(n, m, o) > 1:
            print(f"{predict_rank(n, m, o)} > 1")

In [5]:
load("generate_macaulay.sage")
def test(educational_implementation, prune, can_print):
    q = 16
    v, o, m = 8, 6, 8
    test_it = 1
    n = v + o

    wedge_party(q, v, o, m, test_it, prune, educational_implementation, can_print)
    # test = wedge_once(n, m, o, q, educational_implementation, can_print, prune)

test(False, True, True)

prune? 	 True
Time to calculate the lookup dict: 	 	 0.002350330352783203
Time to calculate the idx and row dicts 	 0.001325368881225586
Time to compute the col_index: 	 	 0.006497621536254883
time to generate Macaulay matrix: 	 0.6819908618927002
M has dimensions: 	 3003 3003


KeyboardInterrupt: 

## Varia

In [ ]:
def rank_via_hilbert_series(pk, q):
    """
    computes the o-th term of the relevant hilbert series using built-in functions. Not yet fully implemented.
    """
    P = PolynomialRing(GF(q), pk[0].nrows(), 'x', order='deglex')
    x_vec = vector(P.gens())
    
    # create ideal
    I = Ideal([x_vec*pk[i]*x_vec for i in range(len(pk))])
    
    # compute its hilbert series
    hs = I.hilbert_series()
    return hs
    
pk, sk, P, oil, vinegar, O, V, gens_vector = generate_UOV_variables(n, m, o, q)
n, m, o = 5, 2, 2
rank_via_hilbert_series(pk, q)
bool_val = is_admissible(n, m, o)
print(bool_val)

## Testen

### `col_index()`

In [ ]:
# TEST create_col_index
load("generate_macaulay.sage")

n, m, o, q = 5, 2, 2, 2
def test_create_col_index(n, m, o, q):
    F = GF(q)
    d = o
    R = PolynomialRing(F, n, 'x')
    print(create_col_index(R.gens(), n, d))

test_create_col_index(n, m, o, q)

### `generate_Macaulay()`

In [ ]:
load("generate_macaulay.sage")

m, n, o, q = 2, 8, 4, 2
def test(m, n, o, q, can_print=True):
    d = o
    p = []
    order = 'deglex'
    for i in range(m):
        p.append(generate_f(n, o, q))
    res = generate_M(n, q, p, d, order)
    if can_print:
        print(f"Macaulay matrix: \n{res[0]}")
        print('----------------'*4)
        print(f"row index: \t {res[1]}")
        print('----------------'*4)
        print(f"column index: \t {res[2]}")

test(m, n, o, q)

In [ ]:
admissible_parameters = [ # v, o, m
    [4, 3, 10],
    [4, 3, 5], # geeft vaak stelsels waarvan de Macaulay-matrix een dimensie groter dan 1 heeft
    [4, 3, 6], # voor m = 6 lijkt dat opgelost
    [9, 8, 5],
    [9, 6, 6],
    [10, 6 , 6],
    [11, 7, 7],
    [12, 8, 8],
    [],
    []
]